# Initialize Spark Environment

This notebook will introduce you to some pre-processing examples to get your dataset in the right format for the stock prediction group assignment.

#### 1. Initialize PySpark & import packages

In [1]:
# import findspark
import findspark
# initialize findspark with spark directory
findspark.init()
# import pyspark
import pyspark
# create spark context
sc = pyspark.SparkContext()
# create spark session
spark = pyspark.sql.SparkSession(sc)

In [2]:
# inspect spark session
spark

In [3]:
# import python packages
import os
import pandas as pd

# import pyspark functionalities
from pyspark.sql import functions as F
from pyspark.sql import types as T
from pyspark.sql.window import Window

# turn of warnings
import warnings
warnings.filterwarnings("ignore")

#### 2. Clean up input files & write to parquet format

In [4]:
# define folder where your (stock) data is stored
directory = './Data/'

# look for a specific stock by using its ticker
TICKER = 'CGC'

In [5]:
# get all files in directory
all_files = os.listdir(directory)
all_files

['Canadian_gold_corp_announcements.csv',
 'licensed-image.jfif',
 'Price History_CGC_20251010_1220.xlsx',
 'Price History_CGC_20251010_1221.xlsx',
 'Price History_CGC_20251010_1223.xlsx',
 'Price History_CGC_20251010_1224.xlsx']

In [6]:
# only keep files that include ticker of the stock
all_files = [directory + file for file in all_files if TICKER in file]
all_files

['./Data/Price History_CGC_20251010_1220.xlsx',
 './Data/Price History_CGC_20251010_1221.xlsx',
 './Data/Price History_CGC_20251010_1223.xlsx',
 './Data/Price History_CGC_20251010_1224.xlsx']

In [7]:
# create a python function to read excel (.xlsx) files 
# NOTE: we use pandas here because pyspark cannot read excel files
def load_excel_file(filepath):
    # Load entire excel file to inspect structure
    df_temp = pd.read_excel(filepath, sheet_name=0) 
    df_temp.columns = [i for i in range(df_temp.shape[1])]

    # Find 'Local Date' in columns 0 or 2 to determine file structure
    for col in [0, 2]:
        if col < len(df_temp.columns) and (df_temp[col] == 'Local Date').any():
            header_row = (df_temp[col] == 'Local Date').idxmax() + 1
            usecols = 'A:I' if col == 0 else 'C:K'  # Skip exchange columns if present
            break
    else:
        raise ValueError("Could not find 'Local Date' column")
    
    # Read actual data and convert to Spark DataFrame
    df = pd.read_excel(filepath, sheet_name=0, header=header_row, usecols=usecols)
    return spark.createDataFrame(df)

In [8]:
# read in a single file
df_1 = load_excel_file(all_files[0])
df_1.show()

+-------------------+-------------------+-----+-----+------------------+-----+-----+-----+------+
|         Local Date|         Local Time|Close|  Net|              %Chg| Open|  Low| High|Volume|
+-------------------+-------------------+-----+-----+------------------+-----+-----+-----+------+
|2025-01-10 22:31:00|2025-01-10 22:31:00|  NaN|  NaN|               NaN|  NaN|  NaN|  NaN|   NaN|
|2025-01-10 22:01:00|2025-01-10 22:01:00|  NaN|  NaN|               NaN|  NaN|  NaN|  NaN|   NaN|
|2025-01-10 22:00:00|2025-01-10 22:00:00|  NaN|  NaN|               NaN|  NaN|  NaN|  NaN|   NaN|
|2025-01-10 21:54:00|2025-01-10 21:54:00|0.155|  0.0|               0.0|0.155|0.155|0.155|5000.0|
|2025-01-10 17:19:00|2025-01-10 17:19:00|  NaN|  NaN|               NaN|  NaN|  NaN|  NaN|   NaN|
|2025-01-10 17:16:00|2025-01-10 17:16:00|  NaN|  NaN|               NaN|  NaN|  NaN|  NaN|   NaN|
|2025-01-10 16:24:00|2025-01-10 16:24:00|  NaN|  NaN|               NaN|  NaN|  NaN|  NaN|   NaN|
|2025-01-10 16:23:00

In [9]:
# create a function to loop over all relevant excel files in directory
# put all data in one large PySpark dataframe
def load_ticker_files(base_dir, ticker):
    
    # placeholder for the final PySpark DataFrame
    complete_data_df = None

    # loop through all files in the directory
    for filename in os.listdir(base_dir):
        if filename.startswith(f"Price History_{ticker}_") and filename.endswith(".xlsx"):
            filepath = os.path.join(base_dir, filename)
            
            # load the excel file and convert to spark dataframe
            spark_df = load_excel_file(filepath)

            # initialize dataframe or append the data to existing dataframe
            if complete_data_df is None:
                # if first iteration, initialize dataframe
                complete_data_df = spark_df 
            else:
                # append new data to existing dataframe
                complete_data_df = complete_data_df.union(spark_df)

    return complete_data_df

Let's put everything together:

In [10]:
# read in all files using our functions
df_init = load_ticker_files(directory, TICKER)

# create spark dataframe
df_init.show()

+-------------------+-------------------+-----+-----+------------------+-----+-----+-----+------+
|         Local Date|         Local Time|Close|  Net|              %Chg| Open|  Low| High|Volume|
+-------------------+-------------------+-----+-----+------------------+-----+-----+-----+------+
|2025-01-10 22:31:00|2025-01-10 22:31:00|  NaN|  NaN|               NaN|  NaN|  NaN|  NaN|   NaN|
|2025-01-10 22:01:00|2025-01-10 22:01:00|  NaN|  NaN|               NaN|  NaN|  NaN|  NaN|   NaN|
|2025-01-10 22:00:00|2025-01-10 22:00:00|  NaN|  NaN|               NaN|  NaN|  NaN|  NaN|   NaN|
|2025-01-10 21:54:00|2025-01-10 21:54:00|0.155|  0.0|               0.0|0.155|0.155|0.155|5000.0|
|2025-01-10 17:19:00|2025-01-10 17:19:00|  NaN|  NaN|               NaN|  NaN|  NaN|  NaN|   NaN|
|2025-01-10 17:16:00|2025-01-10 17:16:00|  NaN|  NaN|               NaN|  NaN|  NaN|  NaN|   NaN|
|2025-01-10 16:24:00|2025-01-10 16:24:00|  NaN|  NaN|               NaN|  NaN|  NaN|  NaN|   NaN|
|2025-01-10 16:23:00

In [11]:
# check data types
df_init.printSchema()

root
 |-- Local Date: timestamp (nullable = true)
 |-- Local Time: timestamp (nullable = true)
 |-- Close: double (nullable = true)
 |-- Net: double (nullable = true)
 |-- %Chg: double (nullable = true)
 |-- Open: double (nullable = true)
 |-- Low: double (nullable = true)
 |-- High: double (nullable = true)
 |-- Volume: double (nullable = true)



Because we don't want to perform the read action everytime we want to manipulate the data it might be a good idea to cache() the spark dataframe to memory for quicker access. **NOTE**: you can already add this step within your function.

In [12]:
# cache the spark datafrome to memory
df_init = df_init.cache()

# call an action to perform
df_init.show()

+-------------------+-------------------+-----+-----+------------------+-----+-----+-----+------+
|         Local Date|         Local Time|Close|  Net|              %Chg| Open|  Low| High|Volume|
+-------------------+-------------------+-----+-----+------------------+-----+-----+-----+------+
|2025-01-10 22:31:00|2025-01-10 22:31:00|  NaN|  NaN|               NaN|  NaN|  NaN|  NaN|   NaN|
|2025-01-10 22:01:00|2025-01-10 22:01:00|  NaN|  NaN|               NaN|  NaN|  NaN|  NaN|   NaN|
|2025-01-10 22:00:00|2025-01-10 22:00:00|  NaN|  NaN|               NaN|  NaN|  NaN|  NaN|   NaN|
|2025-01-10 21:54:00|2025-01-10 21:54:00|0.155|  0.0|               0.0|0.155|0.155|0.155|5000.0|
|2025-01-10 17:19:00|2025-01-10 17:19:00|  NaN|  NaN|               NaN|  NaN|  NaN|  NaN|   NaN|
|2025-01-10 17:16:00|2025-01-10 17:16:00|  NaN|  NaN|               NaN|  NaN|  NaN|  NaN|   NaN|
|2025-01-10 16:24:00|2025-01-10 16:24:00|  NaN|  NaN|               NaN|  NaN|  NaN|  NaN|   NaN|
|2025-01-10 16:23:00

In [13]:
# sort ticker data based on registration date (minute-per-minute)
df_init = df_init.sort('Local Date', ascending=True)

# show
df_init.show()

+-------------------+-------------------+-----+------+-------------------+-----+-----+-----+-------+
|         Local Date|         Local Time|Close|   Net|               %Chg| Open|  Low| High| Volume|
+-------------------+-------------------+-----+------+-------------------+-----+-----+-----+-------+
|2024-10-10 10:01:00|2024-10-10 10:01:00|  NaN|   NaN|                NaN|  NaN|  NaN|  NaN|    NaN|
|2024-10-10 13:31:00|2024-10-10 13:31:00|  NaN|   NaN|                NaN|  NaN|  NaN|  NaN|    NaN|
|2024-10-10 13:32:00|2024-10-10 13:32:00|  NaN|   NaN|                NaN|  NaN|  NaN|  NaN|    NaN|
|2024-10-10 15:31:00|2024-10-10 15:31:00|0.195|   NaN|                NaN|0.195|0.195|0.195|  500.0|
|2024-10-10 16:22:00|2024-10-10 16:22:00|  NaN|   NaN|                NaN|  NaN|  NaN|  NaN|    NaN|
|2024-10-10 16:48:00|2024-10-10 16:48:00| 0.19|-0.005|-0.0256410256410257| 0.19| 0.19| 0.19|20000.0|
|2024-10-10 16:49:00|2024-10-10 16:49:00|  NaN|   NaN|                NaN|  NaN|  NaN|  NaN

In [14]:
# let's check the minimum and maximum date
min_date = df_init.select(F.min('Local Date')).collect()[0][0]
print('minimum date in dataframe:', min_date)

max_date = df_init.select(F.max('Local Date')).collect()[0][0]
print('maximum date in dataframe:', max_date)

minimum date in dataframe: 2024-10-10 10:01:00
maximum date in dataframe: 2025-10-10 10:01:00
maximum date in dataframe: 2025-10-10 10:01:00


#### Date format

As you explore the data further, you'll notice quite a lot of time jumps. So we need to come up with a logic for this. Let's inspect the **data for a normal weekday**, for example Thursday October 10th, 2024:

In [15]:
# subset spark dataframe for all dates have YYYY-MM-DD equal to 2024-10-10 (first)
filtered_df = df_init.filter(F.to_date(df_init['Local Date']) == '2024-10-10') \
                     .sort('Local Date', ascending=True)

# show
filtered_df.show()

+-------------------+-------------------+-----+------+-------------------+-----+-----+-----+-------+
|         Local Date|         Local Time|Close|   Net|               %Chg| Open|  Low| High| Volume|
+-------------------+-------------------+-----+------+-------------------+-----+-----+-----+-------+
|2024-10-10 10:01:00|2024-10-10 10:01:00|  NaN|   NaN|                NaN|  NaN|  NaN|  NaN|    NaN|
|2024-10-10 13:31:00|2024-10-10 13:31:00|  NaN|   NaN|                NaN|  NaN|  NaN|  NaN|    NaN|
|2024-10-10 13:32:00|2024-10-10 13:32:00|  NaN|   NaN|                NaN|  NaN|  NaN|  NaN|    NaN|
|2024-10-10 15:31:00|2024-10-10 15:31:00|0.195|   NaN|                NaN|0.195|0.195|0.195|  500.0|
|2024-10-10 16:22:00|2024-10-10 16:22:00|  NaN|   NaN|                NaN|  NaN|  NaN|  NaN|    NaN|
|2024-10-10 16:48:00|2024-10-10 16:48:00| 0.19|-0.005|-0.0256410256410257| 0.19| 0.19| 0.19|20000.0|
|2024-10-10 16:49:00|2024-10-10 16:49:00|  NaN|   NaN|                NaN|  NaN|  NaN|  NaN

Are there certain hours for which we don't have data?

In [16]:
# extract hour from timestamp and count records for each hour
hourly_data = filtered_df.withColumn('hour', F.hour(filtered_df['Local Date'])) \
                .groupBy('hour') \
                .count() \
                .orderBy('hour')

# show
hourly_data.show()

+----+-----+
|hour|count|
+----+-----+
|  10|    1|
|  13|    2|
|  15|    1|
|  16|    3|
|  17|    1|
|  18|    1|
|  20|    1|
|  21|    3|
|  22|    2|
+----+-----+



As you can see, the timestamps of the stock data is not at all what we would expect. Usually, the stock exchange operates **from 9:30 AM until 4:00 PM**. This clearly doesn't align with our data going well past midnight and **a huge gap in the middle of the night** between 10:00 PM and 10:00 AM. **What could be a possible explanation for this?**

The answer has to do with the dataset collection process. We collected the data here in Belgium, however we are working with data from a company that's noted on the **Toronto Venture Exchange (TSXV)**. Therefore, we should use the local time of Toronto instead of Brussels. There still are some observations outside regular trading hours but these are NA for this day. However, it could still be that there are real trades outside the official trading hours:

- **Pre Opening Session**: 7:00 AM to 9:30 AM ET
- **Core Trading Session**: from 9:30 AM to 4:00 PM ET 
- **Extended Hours**: 4:00 PM to 5:00 PM ET

If we translate this to Central European Time (CET):

- **Pre Opening Session**: 1:00 PM to 3:30 PM CET
- **Core Trading Session**: from 3:30 PM to 10:00 PM CET 
- **Extended Hours**: 10:00 PM to 11:00 PM CET

So, this is the reason why we could have a gap between 23:00 PM and 7:00 AM each time...

More information: https://www.tsx.com/en/trading/calendars-and-trading-hours/trading-hours

<img src="./Data/licensed-image.jfif" alt="Time Zones" width="1000"/>

Let's fix this by converting the timezone **from CET** (Belgium) **to ET** (Toronto):

``HINT``: To be able to reuse functionalities in different scenarios more easily without having to rewrite all transformations over and over again, consider creating pyspark / UDF functions to invoke

In [17]:
# overwrite the Local Date column with ET time instead of CET 
df_with_et = df_init.withColumn("Local Date (ET)",
                                F.from_utc_timestamp(
                                    F.to_utc_timestamp(F.col("Local Date"), "CET"), 
                                    "America/New_York")) \
                    .withColumnRenamed("Local Time", "Local Date (CET)") \
                    .drop("Local Date")

# compare with Local Time column (should be )
df_with_et.show()

+-------------------+-----+------+-------------------+-----+-----+-----+-------+-------------------+
|   Local Date (CET)|Close|   Net|               %Chg| Open|  Low| High| Volume|    Local Date (ET)|
+-------------------+-----+------+-------------------+-----+-----+-----+-------+-------------------+
|2024-10-10 10:01:00|  NaN|   NaN|                NaN|  NaN|  NaN|  NaN|    NaN|2024-10-10 04:01:00|
|2024-10-10 13:31:00|  NaN|   NaN|                NaN|  NaN|  NaN|  NaN|    NaN|2024-10-10 07:31:00|
|2024-10-10 13:32:00|  NaN|   NaN|                NaN|  NaN|  NaN|  NaN|    NaN|2024-10-10 07:32:00|
|2024-10-10 15:31:00|0.195|   NaN|                NaN|0.195|0.195|0.195|  500.0|2024-10-10 09:31:00|
|2024-10-10 16:22:00|  NaN|   NaN|                NaN|  NaN|  NaN|  NaN|    NaN|2024-10-10 10:22:00|
|2024-10-10 16:48:00| 0.19|-0.005|-0.0256410256410257| 0.19| 0.19| 0.19|20000.0|2024-10-10 10:48:00|
|2024-10-10 16:49:00|  NaN|   NaN|                NaN|  NaN|  NaN|  NaN|    NaN|2024-10-10 

However, **we still have time jumps** in the data. This can be due to there being no trades during certain periods, and because of weekends or holidays. As we will later aggregate the data on a biweekly level, we can simply ignore observations without trades and with missing values.

In [18]:
# Remove rows containing NaN values in volume column
df_with_et = df_with_et.filter((F.col('Volume').isNotNull()) & (~F.isnan(F.col('Volume')))) \
    .select('Local Date (CET)', 'Local Date (ET)', 'Close', 'Net', '%Chg', 'Open', 'High', 'Low', 'Volume')
# Show the cleaned DataFrame
df_with_et.show()

+-------------------+-------------------+-----+------+-------------------+-----+-----+-----+-------+
|   Local Date (CET)|    Local Date (ET)|Close|   Net|               %Chg| Open| High|  Low| Volume|
+-------------------+-------------------+-----+------+-------------------+-----+-----+-----+-------+
|2024-10-10 15:31:00|2024-10-10 09:31:00|0.195|   NaN|                NaN|0.195|0.195|0.195|  500.0|
|2024-10-10 16:48:00|2024-10-10 10:48:00| 0.19|-0.005|-0.0256410256410257| 0.19| 0.19| 0.19|20000.0|
|2024-10-10 21:57:00|2024-10-10 15:57:00|0.195| 0.005| 0.0263157894736842|0.195|0.195|0.195|20500.0|
|2024-10-10 22:00:00|2024-10-10 16:00:00| 0.19|-0.005|-0.0256410256410257| 0.19| 0.19| 0.19| 5000.0|
|2024-10-15 15:31:00|2024-10-15 09:31:00| 0.19|   0.0|                0.0| 0.19| 0.19| 0.19| 5000.0|
|2024-10-15 16:09:00|2024-10-15 10:09:00| 0.19|   0.0|                0.0| 0.19| 0.19| 0.19| 2500.0|
|2024-10-15 21:45:00|2024-10-15 15:45:00|  0.2|  0.01| 0.0526315789473685|0.195|  0.2|0.195

### Write data to disk
After you preprocessed your data, you can write it to an efficient storage format file (like a Parquet or Delta file). As we are working with big chunks of data, it's best practice to use these file formats to optimize reading, writing and searching mechanisms. 

In [19]:
# file path to write to
parquet_path = "./Data/Cleaned Data/processed_data.parquet"

# write the dataframe to parquet format (delta is also fine if you want more version control):
df_with_et.write.parquet(parquet_path, mode='overwrite')

In [20]:
# NOTE: If you have a lot of data, it's a good practice to partition your data, for example by using the dates as a partitioning key
#df_full.write.partitionBy("Date").parquet(parquet_path, mode="overwrite")

In [21]:
# file path to read from
parquet_read_path = "./Data/Cleaned Data/processed_data.parquet"

# Read the Parquet file
parquet_df = spark.read.parquet(parquet_read_path)

# show
parquet_df.show()

+-------------------+-------------------+-----+------+-------------------+-----+-----+-----+-------+
|   Local Date (CET)|    Local Date (ET)|Close|   Net|               %Chg| Open| High|  Low| Volume|
+-------------------+-------------------+-----+------+-------------------+-----+-----+-----+-------+
|2024-10-10 15:31:00|2024-10-10 09:31:00|0.195|   NaN|                NaN|0.195|0.195|0.195|  500.0|
|2024-10-10 16:48:00|2024-10-10 10:48:00| 0.19|-0.005|-0.0256410256410257| 0.19| 0.19| 0.19|20000.0|
|2024-10-10 21:57:00|2024-10-10 15:57:00|0.195| 0.005| 0.0263157894736842|0.195|0.195|0.195|20500.0|
|2024-10-10 22:00:00|2024-10-10 16:00:00| 0.19|-0.005|-0.0256410256410257| 0.19| 0.19| 0.19| 5000.0|
|2024-10-15 15:31:00|2024-10-15 09:31:00| 0.19|   0.0|                0.0| 0.19| 0.19| 0.19| 5000.0|
|2024-10-15 16:09:00|2024-10-15 10:09:00| 0.19|   0.0|                0.0| 0.19| 0.19| 0.19| 2500.0|
|2024-10-15 21:45:00|2024-10-15 15:45:00|  0.2|  0.01| 0.0526315789473685|0.195|  0.2|0.195

#### 3. Inspect event file
This file contains information about certain events that happened in the company. These are manually collected from the company news website, but you could also use web scraping to get this information automatically, the choice is up to you. Just be careful when creating the **dependent 'financing_announcement' event**, as the news website might also show e.g. closing of private placements, which should not be considered as the dependent, only the announcement of new financings.

You can also add other events to this file to be used as features in your model, for example drilling results, property acquisitions,... and even use the full news articles as text data for NLP techniques later on.

In [22]:
# Load and inspect event file
event_df = spark.read.csv('./Data/Canadian_gold_corp_announcements.csv', header=True, inferSchema=True)
event_df.show()

+-------------------+------+----------+--------------------+
|         stock_name|ticker| date-time|                type|
+-------------------+------+----------+--------------------+
|Canadian Gold Corp.|   CGC|2025-06-19|financing_announc...|
|Canadian Gold Corp.|   CGC|2025-06-10| exploration_results|
|Canadian Gold Corp.|   CGC|2025-06-02|property_acquisition|
|Canadian Gold Corp.|   CGC|2025-05-16|property_acquisition|
|Canadian Gold Corp.|   CGC|2025-05-12| exploration_results|
|Canadian Gold Corp.|   CGC|2025-03-10|financing_announc...|
|Canadian Gold Corp.|   CGC|2025-03-03| exploration_results|
|Canadian Gold Corp.|   CGC|2025-02-18| exploration_results|
|Canadian Gold Corp.|   CGC|2025-02-10| exploration_results|
|Canadian Gold Corp.|   CGC|2024-11-20|financing_announc...|
|Canadian Gold Corp.|   CGC|2024-11-14| exploration_results|
|Canadian Gold Corp.|   CGC|2024-10-16| exploration_results|
+-------------------+------+----------+--------------------+



In [23]:
# let's check how many financing announcements we have
dependent_df = event_df.filter(F.col('type') == 'financing_announcement')
dependent_df.show()

+-------------------+------+----------+--------------------+
|         stock_name|ticker| date-time|                type|
+-------------------+------+----------+--------------------+
|Canadian Gold Corp.|   CGC|2025-06-19|financing_announc...|
|Canadian Gold Corp.|   CGC|2025-03-10|financing_announc...|
|Canadian Gold Corp.|   CGC|2024-11-20|financing_announc...|
+-------------------+------+----------+--------------------+



In [24]:
# What other types of announcements do we have?
event_df.groupBy('type').count().show()

+--------------------+-----+
|                type|count|
+--------------------+-----+
|property_acquisition|    2|
|financing_announc...|    3|
| exploration_results|    7|
+--------------------+-----+

